## 4. Exploratory Data Analysis (EDA) e Validazione Statistica

All'inizio della pipeline di preprocessing, il dataset `XAUUSD_ReadyToUse.csv` si presenta come una matrice densa e continua, pronta per la modellazione. Tuttavia, prima di procedere con l'implementazione degli algoritmi predittivi (Pattern Matching e Machine Learning), è utile condurre un'analisi esplorativa dei dati (EDA) per estrarre le proprietà statistiche del sottostante, valutare la dimensionalità delle feature e verificare l'assenza di anomalie macroscopiche prima della pulizia.

Lo script (`EDA.py`) esplora i dati concentrandosi su tre aspetti chiave:

### 4.1 Statistiche Descrittive e Filtro delle Anomalie Sistemiche
Prima di qualsiasi computazione statistica, il modulo EDA applica un filtro logico essenziale: l'esclusione temporanea delle righe contrassegnate dalla flag `Missing = True`. 
Poiché lo Step 4 della pipeline ha iniettato valori vuoti (`NaN`) per mantenere la continuità assoluta dell'asse temporale durante i weekend, includere questi "macro-gap" nel calcolo delle medie o delle varianze introdurrebbe un grave bias statistico (sottostima artificiale della volatilità). Le metriche descrittive (media, deviazione standard, minimi e massimi) vengono calcolate escludendo le pause del fine settimana per non falsare l'analisi dei volumi.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

"""
=========================================================================================
FILE: EDA.py (Exploratory Data Analysis)

OBIETTIVO: 
Esplorare visivamente e statisticamente il dataset pulito prima della fase di modellazione.
Questo script verifica le distribuzioni delle feature, analizza la stagionalità intraday 
del mercato e controlla la correlazione tra le variabili di prezzo.

FUNZIONAMENTO:
- Carica il dataset ML-Ready.
- Filtra le righe contrassegnate come 'Missing' (es. chiusure weekend) per non distorcere le statistiche.
- Stampa le metriche descrittive principali (Media, Dev.Std, Min, Max, Quartili).
- Genera un pannello di istogrammi per analizzare la distribuzione di OHLC e TotalTicks.
- Aggrega il volume (TotalTicks) per ora solare, evidenziando i picchi di volatilità (es. sessione NY/London).
- Genera e plotta una Matrice di Correlazione per le feature numeriche.

INPUT:  ReadyData/XAUUSD_ReadyToUse1.csv
=========================================================================================
"""

# Configurazione dinamica dei percorsi
file_path = os.path.join(os.getcwd(), 'Data Management', 'ReadyData', 'XAUUSD_ReadyToUse.csv')

# Caricamento del dataset
print("Caricamento del dataset in corso...")
df = pd.read_csv(file_path, parse_dates=['Datetime'])
print(df.head())

# Esclusione dei macro-gap (weekend) dall'analisi statistica
df_clean = df[df['Missing'] == False].copy()

# Statistiche descrittive
print("\n--- STATISTICHE DESCRITTIVE ---")
print(df[['Open', 'High', 'Low', 'Close', 'TotalTicks']].describe())

### 4.2 Analisi Distribuzionale (Feature Spaziali)
La distribuzione spaziale delle variabili di prezzo (Open, High, Low, Close) e di volume (TotalTicks) viene osservata visivamente tramite un pannello di istogrammi. 
Questo passaggio permette di identificare:
* L'assenza di valori *outlier* anomali derivanti da errori del data provider (es. flash crash inesistenti a valore 0).
* La forte asimmetria tipica dei volumi tick-by-tick, dominati da micro-variazioni (candele a basso volume) alternate a rari e violenti *spike* di liquidità (oltre 5000 tick/minuto), solitamente coincidenti con rilasci macroeconomici.

In [ ]:
# ==========================================
# 1. DISTRIBUZIONE DELLE FEATURE (Dashboard)
# ==========================================
# Creazione di un'unica finestra pulita con 5 grafici allineati
fig, ax = plt.subplots(ncols=5, figsize=(20, 5))
fig.suptitle("Distribuzione Statistica: Prezzi (OHLC) e Volumi (TotalTicks)", fontsize=16, fontweight='bold')

df_clean.hist("Open", bins=100, ax=ax[0], edgecolor='black', color='skyblue')
df_clean.hist("High", bins=100, ax=ax[1], edgecolor='black', color='lightgreen')
df_clean.hist("Low", bins=100, ax=ax[2], edgecolor='black', color='salmon')
df_clean.hist("Close", bins=100, ax=ax[3], edgecolor='black', color='gold')
df_clean.hist("TotalTicks", bins=100, ax=ax[4], edgecolor='black', color='purple')

# Miglioroamento della spaziatura tra i grafici
plt.tight_layout()
plt.show()

### 4.3 Stagionalità Intraday e Volatilità
Le serie storiche finanziarie, in particolare l'asset XAUUSD, presentano una marcata ciclicità legata agli orari operativi delle piazze globali. L'algoritmo raggruppa i dati per ora solare (0-23) e ne calcola la deviazione standard del parametro `TotalTicks`. 
Questa metrica funge da indicatore per la volatilità intraday, permettendo di isolare visivamente i regimi di mercato: l'incremento di volatilità fisiologico durante le sovrapposizioni delle sessioni di Londra e New York (14:00 - 17:00) rispetto alle fasi di lateralità della sessione asiatica.

In [ ]:
# ==========================================
# 2. ANALISI DELLA STAGIONALITÀ INTRADAY
# ==========================================
# Raggruppamento per ora per l'analisi della volatilità (approssimata tramite deviazione standard del volume)
df_clean['Hour'] = df_clean['Datetime'].dt.hour
volatility = df_clean.groupby('Hour')['TotalTicks'].std()

plt.figure(figsize=(12, 6))
volatility.plot(kind='bar', color='coral', edgecolor='black')
plt.title("Volatilità Intraday (Deviazione Standard del TotalTicks per Ora Solare)", fontsize=14)
plt.xlabel("Ora del giorno (0-23)", fontsize=12)
plt.ylabel("Deviazione Standard dei Tick", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=0)
plt.show()

### 4.4 Analisi di Correlazione (Matrice di Pearson)
La matrice calcola la correlazione di Pearson tra le variabili numeriche. Le componenti OHLC mostrano fisiologicamente una correlazione prossima a `1.0`, indicando una totale ridondanza informativa a livello di prezzo assoluto (il prezzo in un minuto si sposta di pochissimo rispetto al suo valore totale). Questa evidenza empirica giustifica il passaggio algoritmico successivo: i modelli non useranno i prezzi assoluti, bensì le loro variazioni relative (feature geometriche come Body e Range), in modo da rendere le variabili indipendenti e stazionarie.

In [ ]:
# ==========================================
# 3. MATRICE DI CORRELAZIONE
# ==========================================
corr_matrix = df_clean[['Open', 'High', 'Low', 'Close', 'TotalTicks']].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='none', vmin=-1, vmax=1)
plt.colorbar()

# Aggiunta etichette degli assi
plt.xticks(range(len(corr_matrix)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix)), corr_matrix.columns)
plt.title("Matrice di Correlazione di Pearson", fontsize=14)

# Ciclo per stampare i valori numerici all'interno dei quadrati
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix.columns)):
        text = plt.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                       ha="center", va="center", color="black" if abs(corr_matrix.iloc[i, j]) < 0.5 else "white")

plt.tight_layout()
plt.show()